# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the **FAIR^2** dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema at the provided URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata as an object for exploration
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their IDs, and the fields in each. All entities are referenced by their `@id` field as per the Croissant schema.

In [ ]:
# List all available RecordSets in the dataset
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")

# For each RecordSet, print its fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet '@id': {rs.id}")
    print(f"Name: {rs.name}")
    print(f"Description: {rs.description}")
    # List fields
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    # List columns (for reference)
    if hasattr(rs, "columns") and rs.columns:
        print("  Columns (by @id):")
        for col in rs.columns:
            print(f"    - @id: {col.id} (name: {col.name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all available record sets as DataFrames, using their `@id` for reference.

In [ ]:
# Extract each record set in the dataset
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet '@id': {record_set_id}  → shape={df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    else:
        print(f"RecordSet '@id': {record_set_id} returned 0 records.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate a common data processing workflow: filter on a numeric field, normalize it, group by a categorical field.

For illustration, we select the main protein abundance RecordSet and operate on the `Coverage` and `Gene names` fields. Please substitute with the appropriate `@id` values for the main protein table and fields as shown in the overview above.

In [ ]:
# ---- Adjust these 3 variables after running the previous cell and identifying correct @ids ----
# Set these to the appropriate @id values for your analysis
# E.g., 'cr:RecordSet_xxx', 'cr:field:Coverage', 'cr:field:Gene_names'
main_record_set_id = None  # Example: 'cr:RecordSet_Human_Proteome'
numeric_field_id = None    # Example: 'cr:field:Coverage'
group_field_id = None      # Example: 'cr:field:Gene_names'
# ----

# Find appropriate defaults if not set
if main_record_set_id is None:
    # Heuristically pick the largest DataFrame as the main table
    if len(dataframes) > 0:
        main_record_set_id = max(dataframes.items(), key=lambda item: item[1].shape[0])[0]

# List candidate numeric fields to help the user choose
if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Columns in main RecordSet ({main_record_set_id}):")
    print(df.columns)

    # Find all numeric columns (float or int dtype)
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print("Numeric field candidates:", numeric_candidates)

    # Assign one if not provided
    if numeric_field_id is None and numeric_candidates:
        numeric_field_id = numeric_candidates[0]

    # Find all string/categorical columns for grouping
    group_candidates = df.select_dtypes(include='object').columns.tolist()
    print("Group-by field candidates:", group_candidates)
    if group_field_id is None and group_candidates:
        group_field_id = group_candidates[0]

    # Now process
    # Filter for values larger than a threshold in the numeric field
    threshold = df[numeric_field_id].mean() if numeric_field_id and numeric_field_id in df.columns else 0
    if numeric_field_id and numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a string/categorical field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print(f"RecordSet {main_record_set_id} not found in dataframes.")

## 5. Visualization
Visualize the distributions of selected numeric fields. For instance, the histogram of the coverage field and a boxplot grouped by the gene name or similar protein property.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id and main_record_set_id in dataframes and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field if available
    if group_field_id and group_field_id in df.columns:
        # Only plot for top N groups for clarity
        top_groups = df[group_field_id].value_counts().index[:8]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field_id} by {group_field_id} (Top 8 groups)')
        plt.show()
else:
    print("Visualization not possible: missing record set or field identifiers.")

## 6. Conclusion
In this notebook, we've used `mlcroissant` to explore the main FAIR^2 protein mass spectrometry dataset. We:
- Loaded metadata and listed all record sets and fields using their `@id` values for robust, schema-based data access.
- Loaded the key protein record set into a DataFrame, filtered, normalized and grouped the data for exploratory analysis.
- Visualized numeric protein properties such as coverage and abundance.
Further analysis can build on these steps to drive new discoveries from mass spectrometry datasets standardized with Croissant.